# Fase 1: Exploración de Datos Global

Siguiendo el flujo de Aurélien Géron, antes de limpiar los datos, necesitamos **descubrir y visualizar** su estado actual de forma global.
El objetivo de este notebook es cargar la totalidad de los datos **archivo por archivo** para descubrir:
1. La incompatibilidad de columnas entre los diferentes orígenes.
2. El estado real de la calidad de los datos (Nulos, Duplicados, Tipos de datos) de forma independiente para cada región y dataset nacional.
3. La distribución estadística global para detectar Outliers reales.

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import sys
sys.path.append('../src')
from config import DIR_SIEN_REGIONAL, FILE_ANEMIA_DA, FILE_TAMIZAJE, REGIONES_SELECCIONADAS

# Configuraciones visuales
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Carga de Datos por Archivo (Validación)
Para identificar si alguna región en específico tiene problemas de calidad de datos, cargaremos y almacenaremos cada archivo en un diccionario independiente en lugar de agruparlos.

In [50]:
dfs = {} # Diccionario para almacenar todos los datasets independientemente

try:
    print("Cargando la totalidad de los datos (esto puede tardar unos segundos)...")
    
    # 1. Cargar las 6 regiones independientemente
    for region in REGIONES_SELECCIONADAS:
        nombre_corto = region.replace('.csv', '')
        dfs[nombre_corto] = pd.read_csv(DIR_SIEN_REGIONAL / region, low_memory=False)
    
    # 2. Cargar datasets Nacionales
    dfs['ANEMIA_DA'] = pd.read_csv(FILE_ANEMIA_DA, sep=';', low_memory=False)
    dfs['TAMIZAJE'] = pd.read_csv(FILE_TAMIZAJE, low_memory=False)
    
    print("\nCarga exitosa. No hay errores de rutas en config.py.\n")
    print("--- TAMAÑOS DE LOS ARCHIVOS INDIVIDUALES ---")
    for nombre, df in dfs.items():
        print(f"{nombre} Shape: {df.shape}")

except FileNotFoundError as e:
    print("\nERROR: No se encontró algún archivo.")
    print(f"Detalle: {e}")
    print("Por favor, verifica que los archivos estén en data/raw/ y las rutas en config.py sean correctas.")
except Exception as e:
    print("\nERROR INESPERADO al intentar cargar los datos.")
    print(f"Detalle: {e}")

Cargando la totalidad de los datos (esto puede tardar unos segundos)...

Carga exitosa. No hay errores de rutas en config.py.

--- TAMAÑOS DE LOS ARCHIVOS INDIVIDUALES ---
Niños SAN MARTIN Shape: (54755, 37)
Niños TACNA Shape: (11143, 37)
Niños PUNO Shape: (44052, 37)
Niños CUSCO Shape: (60830, 37)
Niños LORETO Shape: (63384, 37)
Niños PIURA Shape: (104606, 37)
ANEMIA_DA Shape: (59225, 22)
TAMIZAJE Shape: (5285909, 10)


## 2. Incompatibilidad de Cabeceras
Visualizamos cómo difieren las columnas tomando como ejemplo 1 región vs los Nacionales.

In [51]:
print("\n--- ESTRUCTURA DE 1 REGIÓN (San Martín) ---")
print(dfs['Niños SAN MARTIN'].columns.tolist())

print("\n--- ESTRUCTURA ANEMIA DA ---")
print(dfs['ANEMIA_DA'].columns.tolist())

print("\n--- ESTRUCTURA TAMIZAJE ---")
print(dfs['TAMIZAJE'].columns.tolist())


--- ESTRUCTURA DE 1 REGIÓN (San Martín) ---
['sw', 'Diresa', 'Red', 'Microred', 'EESS', 'Dpto_EESS', 'Prov_EESS', 'Dist_EESS', 'Ubigeo_EESS', 'Renipress', 'Pais', 'Sexo', 'EdadMeses', 'UbigeoPN', 'DepartamentoPN', 'ProvinciaPN', 'DistritoPN', 'CentroPobladoPN', 'Juntos', 'SIS', 'Pin', 'Qaliwarma', 'FechaHemoglobina', 'Cred', 'Suplementacion', 'Consejeria', 'Sesion', 'Hemoglobina', 'FechaAtencion', 'FechaNacimiento', 'UbigeoREN', 'DepartamentoREN', 'ProvinciaREN', 'DistritoREN', 'AlturaREN', 'Hbc', 'Dx_anemia']

--- ESTRUCTURA ANEMIA DA ---
['PK_REGISTRO', 'FECHA_REGISTRO', 'GENERO', 'EDAD_REGISTRO', 'TIPO_EDAD', 'ETNIA', 'F_ATENCION', 'GRADO_SEVERIDAD', 'DIAGNOSTICO', 'TIPO_DIAGNOSTICO', 'DESCRIPCION_FINANCIADOR', 'CANTIDAD', 'PROVINCIA', 'DEPARTAMENTO', 'DISTRITO', 'RED', 'MICRORED', 'NOMBRE_ESTABLECIMIENTO', 'CODIGO_UNICO', 'LONGITUD', 'LATITUD', 'FECHA_CORTE']

--- ESTRUCTURA TAMIZAJE ---
['id_persona', 'Edad', 'Tipo_edad', 'Sexo', 'id_ubigeo', 'Fecha_atencion', 'Etapa', 'Diagnosti

## 3. Análisis de Calidad (Duplicados y Nulos por Archivo)
Revisamos cada archivo para descubrir si algún departamento tiene peor calidad de datos que otro.

In [52]:
print("=== DUPLICADOS POR ARCHIVO ===")
for nombre, df in dfs.items():
    duplicados = df.duplicated().sum()
    print(f"{nombre}: {duplicados} filas duplicadas")

print("\n=== % DE NULOS POR ARCHIVO (Top 3 peores columnas) ===")
for nombre, df in dfs.items():
    nulos_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False).head(3)
    print(f"\n-- {nombre} --")
    print(nulos_pct)

=== DUPLICADOS POR ARCHIVO ===
Niños SAN MARTIN: 84 filas duplicadas
Niños TACNA: 9 filas duplicadas
Niños PUNO: 23 filas duplicadas
Niños CUSCO: 39 filas duplicadas
Niños LORETO: 100 filas duplicadas
Niños PIURA: 94 filas duplicadas
ANEMIA_DA: 0 filas duplicadas
TAMIZAJE: 3 filas duplicadas

=== % DE NULOS POR ARCHIVO (Top 3 peores columnas) ===

-- Niños SAN MARTIN --
sw        0.0
Diresa    0.0
Red       0.0
dtype: float64

-- Niños TACNA --
sw        0.0
Diresa    0.0
Red       0.0
dtype: float64

-- Niños PUNO --
sw        0.0
Diresa    0.0
Red       0.0
dtype: float64

-- Niños CUSCO --
sw        0.0
Diresa    0.0
Red       0.0
dtype: float64

-- Niños LORETO --
sw        0.0
Diresa    0.0
Red       0.0
dtype: float64

-- Niños PIURA --
sw        0.0
Diresa    0.0
Red       0.0
dtype: float64

-- ANEMIA_DA --
PK_REGISTRO       0.0
FECHA_REGISTRO    0.0
GENERO            0.0
dtype: float64

-- TAMIZAJE --
id_ubigeo     0.572257
id_persona    0.000000
Edad          0.000000
dtype: 

## 4. Análisis de la Variable Objetivo (Balance de Clases)
Vemos cómo está balanceada la variable objetivo en cada origen.

In [53]:
for nombre, df in dfs.items():
    print(f"\n--- TARGET EN {nombre} ---")
    # Identificar el nombre correcto del target
    target_col = 'Dx_anemia' if 'Dx_anemia' in df.columns else ('DIAGNOSTICO' if 'DIAGNOSTICO' in df.columns else 'Diagnostico')
    print(df[target_col].value_counts(dropna=False, normalize=True).head(3) * 100)


--- TARGET EN Niños SAN MARTIN ---
Dx_anemia
Normal             85.352936
Anemia Leve        13.385079
Anemia Moderada     1.249201
Name: proportion, dtype: float64

--- TARGET EN Niños TACNA ---
Dx_anemia
Normal             81.566903
Anemia Leve        16.081845
Anemia Moderada     2.315355
Name: proportion, dtype: float64

--- TARGET EN Niños PUNO ---
Dx_anemia
Normal             84.295832
Anemia Leve        12.841642
Anemia Moderada     2.760374
Name: proportion, dtype: float64

--- TARGET EN Niños CUSCO ---
Dx_anemia
Normal             83.741575
Anemia Leve        13.046194
Anemia Moderada     3.154693
Name: proportion, dtype: float64

--- TARGET EN Niños LORETO ---
Dx_anemia
Normal             87.346965
Anemia Leve         9.499243
Anemia Moderada     3.106462
Name: proportion, dtype: float64

--- TARGET EN Niños PIURA ---
Dx_anemia
Normal             91.880963
Anemia Leve         6.619123
Anemia Moderada     1.487486
Name: proportion, dtype: float64

--- TARGET EN ANEMIA_DA ---


## 5. Estadística Descriptiva (Detección de Outliers)
Revisamos los estadísticos principales de TODAS las variables numéricas por cada archivo, excluyendo los IDs (que no aportan valor estadístico).

In [54]:
for nombre, df in dfs.items():
    print(f"\n=== ESTADÍSTICAS EN {nombre} ===")
    
    # Excluimos columnas que son claramente IDs (no tiene sentido calcular el 'promedio' de un ID)
    cols_a_ignorar = [c for c in df.columns if c.lower().startswith('id_') or c == 'PK_REGISTRO' or c == 'sw']
    df_stats = df.drop(columns=cols_a_ignorar, errors='ignore')
    
    # Pandas .describe() automáticamente solo evaluará las columnas numéricas restantes
    display(df_stats.describe())


=== ESTADÍSTICAS EN Niños SAN MARTIN ===


,Ubigeo_EESS,Renipress,EdadMeses,Cred,Suplementacion,Consejeria,Sesion,Hemoglobina,UbigeoREN,AlturaREN,Hbc
count,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000,54755.000000
mean,220575.786376,7243.226500,30.469857,0.671592,0.814976,0.007652,0.000018,11.722992,220575.786376,526.877637,11.509354
std,311.339286,4220.111001,15.205753,0.469639,0.388321,0.087143,0.004274,0.792362,311.339286,285.342226,0.809619
min,220101.000000,6270.000000,6.010000,0.000000,0.000000,0.000000,0.000000,6.600000,220101.000000,0.000000,6.029201
25%,220301.000000,6335.000000,18.000000,0.000000,1.000000,0.000000,0.000000,11.100000,220301.000000,290.000000,11.000000
50%,220602.000000,6421.000000,30.030000,1.000000,1.000000,0.000000,0.000000,11.700000,220602.000000,418.000000,11.500000
75%,220901.000000,6542.000000,42.410000,1.000000,1.000000,0.000000,0.000000,12.100000,220901.000000,856.000000,12.000000
max,221006.000000,35728.000000,59.990000,1.000000,1.000000,1.000000,1.000000,18.500000,221006.000000,1090.000000,18.144294



=== ESTADÍSTICAS EN Niños TACNA ===


,Ubigeo_EESS,Renipress,EdadMeses,Cred,Suplementacion,Consejeria,Sesion,Hemoglobina,UbigeoREN,AlturaREN,Hbc
count,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000,11143.000000
mean,230117.965539,4946.881181,29.760144,0.824374,0.800054,0.007987,0.003859,11.905512,230117.965539,672.954590,11.508125
std,50.976261,7073.847020,15.331114,0.380519,0.399978,0.089017,0.062003,0.937364,50.976261,517.760722,0.908736
min,230101.000000,2864.000000,6.010000,0.000000,0.000000,0.000000,0.000000,6.900000,230101.000000,42.000000,6.493640
25%,230102.000000,2888.000000,16.100000,1.000000,1.000000,0.000000,0.000000,11.300000,230102.000000,562.000000,10.893640
50%,230108.000000,2892.000000,27.140000,1.000000,1.000000,0.000000,0.000000,11.800000,230108.000000,583.000000,11.473647
75%,230110.000000,2920.000000,41.890000,1.000000,1.000000,0.000000,0.000000,12.500000,230110.000000,603.000000,12.061085
max,230408.000000,34527.000000,59.990000,1.000000,1.000000,1.000000,1.000000,18.300000,230408.000000,3853.000000,17.773647



=== ESTADÍSTICAS EN Niños PUNO ===


...

,Ubigeo_EESS,Renipress,EdadMeses,Cred,Suplementacion,Consejeria,Sesion,Hemoglobina,UbigeoREN,AlturaREN,Hbc
count,104606.000000,104606.000000,104606.000000,104606.000000,104606.000000,104606.000000,104606.000000,104606.00000,104606.000000,104606.000000,104606.000000
mean,200333.141321,3126.827572,29.927465,0.891316,0.773522,0.003747,0.001128,11.98284,200333.154972,394.067969,11.770902
std,236.805046,5020.898518,15.358345,0.311244,0.418554,0.061101,0.033568,0.98634,236.792066,739.497386,0.950087
min,200101.000000,1911.000000,6.010000,0.000000,0.000000,0.000000,0.000000,4.00000,200101.000000,7.000000,2.722698
25%,200111.000000,2021.000000,16.490000,1.000000,1.000000,0.000000,0.000000,11.30000,200111.000000,35.000000,11.100000
50%,200301.000000,2095.000000,30.000000,1.000000,1.000000,0.000000,0.000000,12.00000,200301.000000,66.000000,11.700000
75%,200601.000000,2161.000000,42.320000,1.000000,1.000000,0.000000,0.000000,12.60000,200601.000000,135.000000,12.323604
max,200806.000000,36405.000000,59.990000,1.000000,1.000000,1.000000,1.000000,18.20000,200806.000000,2735.000000,18.000000



=== ESTADÍSTICAS EN ANEMIA_DA ===


,FECHA_REGISTRO,EDAD_REGISTRO,F_ATENCION,CANTIDAD,CODIGO_UNICO,LONGITUD,LATITUD,FECHA_CORTE
count,5.922500e+04,59225.000000,5.922500e+04,59225.000000,59225.000000,59225.000000,59225.000000,59225.0
mean,2.020370e+07,3.481857,2.020370e+07,1.007092,6504.450536,-76.616940,-6.604169,20250424.0
std,2.256667e+04,2.664805,2.256667e+04,0.084913,978.482362,0.386289,0.623823,0.0
min,2.016010e+07,-2.000000,2.016010e+07,1.000000,6270.000000,-77.634017,-8.753550,20250424.0
25%,2.018120e+07,1.000000,2.018120e+07,1.000000,6352.000000,-76.850305,-6.792527,20250424.0
50%,2.020083e+07,3.000000,2.020083e+07,1.000000,6406.000000,-76.577641,-6.485973,20250424.0
75%,2.022091e+07,6.000000,2.022091e+07,1.000000,6494.000000,-76.355165,-6.154090,20250424.0
max,2.025043e+07,80.000000,2.025043e+07,3.000000,31276.000000,-75.593100,-5.639154,20250424.0



=== ESTADÍSTICAS EN TAMIZAJE ===


,Edad,Fecha_atencion,Diagnostico
count,5.285909e+06,5.285909e+06,5.285909e+06
mean,1.183335e+01,2.022178e+07,8.501800e+04
std,1.313877e+01,9.923558e+03,1.866226e-03
min,1.000000e+00,2.019010e+07,8.501800e+04
25%,2.000000e+00,2.021050e+07,8.501800e+04
50%,6.000000e+00,2.023013e+07,8.501800e+04
75%,1.700000e+01,2.023050e+07,8.501800e+04
max,9.900000e+01,2.023073e+07,8.501801e+04


## 6. Conclusiones Reales y Siguientes Pasos

Tras ejecutar este análisis granular en la **totalidad de los datos reales**, el diagnóstico de la calidad de información es el siguiente:

1. **Manejo de Duplicados Crítico**: Se ha confirmado que todas las regiones SIEN (por ejemplo, Loreto con 100 duplicados, Piura con 94, San Martín con 84) tienen múltiples filas 100% duplicadas debido a errores de registro del sistema. Estas deberán eliminarse obligatoriamente con `drop_duplicates()` durante la limpieza.
2. **Calidad de Nulos Sorprendente**: A nivel global, la completitud de los datos es excepcional (0% de nulos en la inmensa mayoría de columnas vitales de las 6 regiones y en `ANEMIA_DA`). El archivo `TAMIZAJE` apenas posee ~0.5% de nulos. Esto significa que la etapa de imputación será muy ligera y no perderemos volumen de registros valiosos.
3. **Alerta Roja: Desbalance de Clases**: El conteo del Target muestra un desbalance agresivo natural. Por ejemplo, en San Martín hay ~46,000 niños sanos frente a solo ~8,000 con algún grado de anemia. Si no tratamos esto (usando técnicas de sobremuestreo como SMOTE o aplicando pesos en el modelo), el algoritmo predecirá "Sano" el 100% del tiempo para maximizar el accuracy. Este es el reto más grande para el diseño del modelo.
4. **Armonización de Target**: Archivos como `ANEMIA_DA` poseen diagnósticos descriptivos muy largos ('ANEMIA POR DEFICIENCIA...'). Durante la limpieza, aplicaremos un diccionario maestro para colapsar todas las variantes (CIE-10, textos y grados) a un solo formato binario consolidado (`0 = Normal`, `1 = Anemia`).

**Siguiente paso:** Iniciar la construcción de `02_preprocesamiento.ipynb`. Teniendo este mapa claro, el primer paso será concatenar los 8 archivos bajo un mismo estándar de cabeceras, eliminar los duplicados mapeados, crear el Target Binario universal y prepararnos para la división de los datos.